# T-condition behavioural screen (Qwen2.5-VL-3B)

Plan: `docs/plan_v2.md` section 4, step 3 -- the cheapest kill-test. **Does an
incongruent note move the model's answer on >= ~50 objects (D41)?**

Thin wrapper: all logic lives in `src/circuit_overlap/`. **[FILL IN]** cells need
information the repo does not have; **[VERIFY]** cells check an assumption before
it can silently corrupt the run.

Dataset: `mgolov/Visual-Counterfact`, `split="color"`, 493 rows. Note its real
columns -- `object`, `correct_answer` (a *stringified list* of acceptable
colours), `incorrect_answer`, `original_image`, `counterfact_image`.

Settings: GPU T4, Internet ON.

## 1. Clone the repo and install

In [ ]:
# [FILL IN] your repo URL (or upload the repo as a Kaggle dataset instead)
REPO_URL = ""
REPO_DIR = "/kaggle/working/research_paper"

import os, subprocess, sys
if REPO_URL and not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
print(subprocess.run(["git","-C",REPO_DIR,"rev-parse","--short","HEAD"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate datasets inflect
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

# Visual-Counterfact is read with an HF token (Lietzow's README: "accessed with
# your HF_TOKEN"). Add it as a Kaggle secret named HF_TOKEN.
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 2. Build the object manifest

The T condition pastes notes on the **original_image** (real colour), and the
note then claims `incorrect_answer`.

`correct_answer` is a stringified list such as `"['brown', 'white']"` -- several
colours can be correct for one object. `parse_color_list` normalises it, and the
scorer treats "correct" as membership in that set. `object` is **not** assumed
unique, so ids get a row suffix while the prompt keeps the bare noun.

In [ ]:
import json, pathlib, inflect
from collections import Counter
from datasets import load_dataset
from circuit_overlap.data import parse_color_list, lietzow_object_name
from circuit_overlap.notes import object_bbox_from_pair

OUT = pathlib.Path("/kaggle/working/vc_color"); (OUT / "images").mkdir(parents=True, exist_ok=True)
_eng = inflect.engine()
is_plural = lambda n: bool(_eng.singular_noun(n))

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
print(ds.column_names, len(ds), "rows")

# Lietzow's loader drops invalid rows silently (contextlib.suppress(ValueError)
# around model_validate). Reproduce the same three rules, but count them.
items, reasons = [], Counter()
for i, row in enumerate(ds):
    try:
        name = lietzow_object_name(row["object"], is_plural=is_plural)
    except ValueError as exc:
        reasons["plural" if "plural" in str(exc) else "extra info"] += 1; continue
    try:
        correct = parse_color_list(row["correct_answer"])
        attack = parse_color_list(row["incorrect_answer"])[0]
    except (ValueError, TypeError):
        reasons["unparseable colours"] += 1; continue
    if attack in correct:                    # radish and spider, per their README
        reasons["colour overlap"] += 1; continue
    original = row["original_image"].convert("RGB")
    # The counterfactual image recolours the object and nothing else, so the
    # diff localises the object for free -- no segmentation model needed.
    bbox = object_bbox_from_pair(original, row["counterfact_image"].convert("RGB"))
    if bbox is None:
        reasons["no object box"] += 1          # kept, but placed by fallback anchor
    img_path = OUT / "images" / f"{name.replace(' ', '_')}_{i:04d}.png"
    original.save(img_path)
    items.append({"object_id": f"{name.replace(' ', '_')}_{i:04d}", "object_name": name,
                  "image": str(img_path), "original_colors": list(correct),
                  "counterfactual_color": attack, "object_bbox": bbox})

MANIFEST = OUT / "manifest.json"
MANIFEST.write_text(json.dumps(items, indent=1))
n_boxed = sum(e["object_bbox"] is not None for e in items)
print(f"kept {len(items)}  |  dropped {dict(reasons)}")
print(f"object box recovered for {n_boxed}/{len(items)} objects")
print("Lietzow report 469 valid objects, 467 after the colour-overlap rule.")

In [ ]:
from circuit_overlap.data import load_manifest, collect_color_vocabulary, check_colors_in_vocabulary

entries = load_manifest(MANIFEST)
vocab_from_data = collect_color_vocabulary(entries)
notes_cfg_raw = json.loads(open(f"{REPO_DIR}/configs/notes.json").read())
print(len(entries), "objects")
print("colours in the data:", vocab_from_data)
print("multi-colour objects:", sum(len(e.original_colors) > 1 for e in entries))
print("not in configs/notes.json placeholder vocabulary:",
      check_colors_in_vocabulary(entries, notes_cfg_raw["color_vocabulary"]))
# run_screen unions these automatically, but paste vocab_from_data into
# configs/notes.json so the committed config matches the data.

## 2b. [VERIFY] The 493-vs-467 question

Lietzow et al. report **469** objects (467 after dropping radish and spider);
this split has **493 rows**. Until that is explained, a head-set mismatch at the
replication gate cannot be attributed to either the dataset or the pipeline
(STATE.md, D20). Three candidate explanations, cheapest first.

In [ ]:
# The cascade above is the answer to the 493-vs-467 question, if the numbers land:
#   493 rows -> minus plural / parenthetical names -> 469 -> minus colour overlap -> 467.
# Lietzow's loader applies exactly these rules (src/vdpo/types/models.py) and
# swallows the failures, so the paper's counts are post-filter.
objs = [str(o).strip().lower() for o in ds["object"]]
dupes = {o: c for o, c in Counter(objs).items() if c > 1}
print(f"rows {len(objs)} | unique object names {len(set(objs))} | repeated {len(dupes)}")
print("  repeated examples:", sorted(dupes.items(), key=lambda kv: -kv[1])[:5])
print("  drop reasons:", dict(reasons), "-> kept", len(items))

# How many objects have more than one acceptable colour? (feeds D29a)
print("objects by number of acceptable colours:",
      dict(sorted(Counter(len(e["original_colors"]) for e in items).items())))

# If the counts still do not reach 469/467, check whether an earlier Hub
# revision differs:
from huggingface_hub import list_repo_commits
for c in list_repo_commits("mgolov/Visual-Counterfact", repo_type="dataset")[:10]:
    print(f"  {str(c.created_at)[:10]}  {c.commit_id[:8]}  {c.title[:60]}")

### 2c. [DEBUG] Why 463 and not 467?

Observed on 2026-10-19: our filters keep **463** (dropping 20 plural, 8 extra
info, 2 colour overlap) where Lietzow report **469** valid and **467** after the
colour-overlap rule. The colour-overlap count matches exactly, so the gap is 4
object names we reject and they keep.

Leading hypothesis: their `_sanitize_object_name` **capitalizes** the name
before the plural test (`singular_noun("Scissors")`), while we test lowercase.
Second hypothesis: `inflect` version — their lockfile pins **7.5.0**.

This cell costs no GPU and decodes no images (`ds["object"]` reads one column).

In [ ]:
from importlib.metadata import version
print("inflect", version("inflect"), "(Lietzow pin: 7.5.0)")

names = [str(o) for o in ds["object"]]          # column access: no image decode
dropped = {"plural": [], "extra info": []}
for raw in names:
    try:
        lietzow_object_name(raw, is_plural=is_plural)
    except ValueError as exc:
        dropped["plural" if "plural" in str(exc) else "extra info"].append(raw)

for k, v in dropped.items():
    print(f"\n{k} ({len(v)}): {sorted(set(v))}")

# Hypothesis 1: does capitalisation change inflect's verdict?
print("\nname                 lower->singular      Capitalized->singular   differs")
flips = 0
for raw in sorted(set(dropped["plural"])):
    low = " ".join(raw.lower().replace("_", " ").split())
    cap = low.capitalize()
    a, b = _eng.singular_noun(low), _eng.singular_noun(cap)
    # Their rule: plural iff singular_noun(cap) is truthy AND != cap
    their_verdict = bool(b) and b != cap
    if not their_verdict:
        flips += 1
    print(f"{low:20s} {str(a):20s} {str(b):22s} {'<-- they KEEP this' if not their_verdict else ''}")
print(f"\n{flips} of {len(set(dropped['plural']))} plurals would be kept under their exact rule.")
print(f"-> would give {len(items) + flips} objects (target: 467)")

## 3. [VERIFY] Look at one attack image before spending GPU time

In [ ]:
from circuit_overlap.notes import render_note, variant_texts, matched_neutral_word
from circuit_overlap.tokens import mask_to_token_indices, token_grid_shape
from circuit_overlap.t_screen import load_notes_config

cfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")
e = entries[0]
img = e.open_image()
print(e.object_name, "| correct:", e.original_colors, "| note says:", e.counterfactual_color)
print("object box:", e.object_bbox)
texts = variant_texts(e.primary_original_color, e.counterfactual_color,
                      matched_neutral_word(e.counterfactual_color, cfg["neutral_words"]))
for spec in cfg["specs"]:
    r = render_note(img, texts["incongruent"], spec, bbox=e.object_bbox)
    occ = r.object_occlusion(e.object_bbox)
    print(f"{spec.name:17s} mode={spec.mode:7s} area={r.area_frac:.3f} "
          f"occlusion={occ:.0%} font={r.font_name} {r.font_size_px}px")
    display(r.image.resize((320, 320)))

# Occlusion is the number to watch: a note covering the object removes the
# colour evidence, so the model following it is absence of evidence, not
# typographic override. Scan a few objects before committing GPU time:
occs = []
for e2 in entries[:30]:
    rr = render_note(e2.open_image(), "blue", cfg["specs"][1], bbox=e2.object_bbox)
    occs.append(rr.object_occlusion(e2.object_bbox))
import numpy as np
print(f"\non_object_medium occlusion over 30 objects: "
      f"median {np.nanmedian(occs):.0%}, max {np.nanmax(occs):.0%}")

In [ ]:
# [VERIFY] mask -> image-token mapping against the processor's real grid.
from transformers import AutoProcessor
from circuit_overlap.t_screen import ScreenConfig

config = ScreenConfig()        # [VERIFY] checkpoint id (STATE.md blocker 2)
processor = AutoProcessor.from_pretrained(config.checkpoint,
                                          min_pixels=config.pixels, max_pixels=config.pixels)
enc = processor(text=["x"], images=[r.image], return_tensors="pt")
print("processor grid_thw (t, h, w in patches):", enc["image_grid_thw"])
print("our merged token grid for that image:", token_grid_shape(*r.image.size[::-1]))
# h/2 and w/2 from the processor must equal our (rows, cols) before any H3 claim.

## 4. [VERIFY] Colour words must be single, distinct first tokens (D26)

In [ ]:
from circuit_overlap.t_screen import color_first_token_ids
ids, collisions = color_first_token_ids(processor.tokenizer, vocab_from_data)
print("first-token ids (both case variants, no leading space, as Lietzow do):")
for c, t in ids.items():
    print(f"  {c:8s} {t}")
print("colours sharing a token id:", collisions)
# A collision makes the logit difference meaningless: drop those colours, or
# the objects using them, and say so in RESULTS.md.

## 5. Smoke run on 5 objects, then the full screen

In [ ]:
from circuit_overlap.t_screen import load_model, run_screen
model, processor = load_model(config)
records, summary = run_screen(entries[:5], model, processor, config, cfg)
for r in records[:6]:
    print(f"{r.layout:13s} {r.object_name:12s} correct={r.original_colors} "
          f"ref={r.reference_color} | neutral: {r.answer_neutral!r} "
          f"| attacked: {r.answer_incongruent!r} "
          f"| shift {(r.delta_neutral - r.delta_incongruent):+.2f}")

In [ ]:
import time
t0 = time.time()
records, summary = run_screen(entries, model, processor, config, cfg)
print(f"{time.time()-t0:.0f}s for {2*len(records)} generations")
{k: v for k, v in summary.items() if k not in ("usable_object_ids", "color_vocabulary")}

## 6. Save the result JSON (download it and commit it to `results/`)

In [ ]:
from dataclasses import asdict
from circuit_overlap.provenance import result_envelope, write_result

envelope = result_envelope(
    experiment="t_behavioural_screen",
    payload={"summary": summary, "records": [asdict(r) for r in records]},
    model="Qwen2.5-VL-3B",
    checkpoint=config.checkpoint,
    seed=config.seed,
    prompt=config.prompt_template,
    n=summary["n_objects"],
    repo_dir=REPO_DIR,
    dataset={"source_id": "mgolov/Visual-Counterfact", "split": "color",
             "n_rows_total": len(ds), "n_after_exclusions": len(entries),
             "excluded": sorted(EXCLUDE)},
    layouts=[s.name for s in cfg["specs"]],
    dtype=config.dtype,
    pixels=config.pixels,
)
path = write_result("/kaggle/working/t_screen_qwen25vl3b.json", envelope)
print(path, summary["n_objects_usable"], "usable objects; gate_passed =", summary["gate_passed"])

## 7. What the outcome means (plan section 4)

| Result | Next |
|---|---|
| `gate_passed = True` (>= 50 usable objects) | Proceed to T-circuit discovery |
| Few flips but `continuous_fallback.n_objects_shifted` healthy | Use the continuous variant: patch on logit-difference shifts without an argmax flip |
| Few flips and little shift | Raise note salience (size, contrast) in `configs/notes.json`, or try PaliGemma-3B as primary |

Then record the number in `docs/RESULTS.md` with model, n, seed, prompt and
commit, and log the 493-row / Lietzow-467 discrepancy in `docs/STATE.md`.